In [56]:
import numpy as np
import pandas as pd
from scipy.special import logit, expit

from tqdm.auto import tqdm
import torch

import pickle
from pathlib import Path

import sys
base_path = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(base_path / '2_Propensities'))

import SASRec_class as sasrec

# 1 Choosing Dataset

In [ ]:
datasets = ['ml-1m', 'steam', 'goodreads', 'ml-10m']
DATASET = datasets[0]

print(f"Using dataset: {DATASET}")
base_artifacts = Path.cwd().resolve().parents[1] / 'CausalI2I_artifacts'

Using dataset: ml-1m


# 2 Loading Dataset and Trained Models

In [58]:
# Load chosen pairs and item dictionary
with open(base_artifacts / 'Chosen_Pairs' / DATASET / 'chosen_pairs_ids.pkl', 'rb') as f:
    chosen_pairs_ids = pickle.load(f)

# Load data and create user-item dictionary
data_path = base_artifacts / 'Datasets' / 'Processed' / DATASET
data = pd.read_csv(
    data_path / 'data_clean.csv'
)
users_dict = data.groupby('user_id')['item_id'].apply(list).to_dict()

In [59]:
# Load SASRec model
model_path = base_artifacts / 'SASRec_Models' / DATASET
with open(model_path / f'init_dict.pkl', 'rb') as f:
    init_dict_loaded = pickle.load(f)
L = init_dict_loaded['max_seq_len']

sasrec_model = sasrec.SASRecTorch(**init_dict_loaded)
sasrec_model.load(model_path / f'sasrec.pt')
sasrec_model.eval()

# Load calibrator parameters
with open(model_path / f'calibrator_params.pkl', 'rb') as f:
    calibrator_params = pickle.load(f)
beta = calibrator_params['beta']
print(f"Caliobrator: {calibrator_params['description']}")

# Load test users
with open(data_path / f'test_users.pkl', 'rb') as f:
    test_users = pickle.load(f)

Model loaded from /home/gouni/CausalI2I_new_artifacts/SASRec_Models/ml-1m/sasrec.pt.
num_items:     3707
max_seq_len:   50
device:        cuda
batch_size:    2048
lr:            0.001
weight_decay:  0.0
num_epochs:    20
saved_at:      2026-07-15 12:01:52
note:          None
Caliobrator: p_calibrated = expit(-6.999 + 0.309 * log(p_hat) + -0.386 * log(1-p_hat))


/home/gouni/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


# 3 Set Windows

In [60]:
k = 10 # size of outcome sub-window

In [61]:
padding_idx = data['item_id'].max() + 1

windows = []
for user_id in tqdm(test_users):
    padded_sequence = [padding_idx] * (L - 1) + users_dict[user_id]
    for i in range(len(padded_sequence) - (L + k)):
        windows.append(padded_sequence[i:i+L+k+1])

windows = np.array(windows)

print(f"Shape of windows: {windows.shape}")

  0%|          | 0/1208 [00:00<?, ?it/s]

Shape of windows: (183965, 61)


# 4 Get $T$, $Y$ And $\pi$

In [62]:
treatment_items = windows[:, L]
print(f"Shape of treatment_items: {treatment_items.shape}")

future_items = windows[:, L+1:]
print(f"Shape of future_items: {future_items.shape}")

item_embeddings = sasrec_model.item_embedding.weight.cpu().detach().numpy()
print(f"Shape of item_embeddings: {item_embeddings.shape}")

Shape of treatment_items: (183965,)
Shape of future_items: (183965, 10)
Shape of item_embeddings: (3708, 50)


In [63]:
As = np.array([pair[0] for pair in chosen_pairs_ids])
unique_As = np.unique(As)
A_to_unique_A_idx = {A: i for i, A in enumerate(unique_As)}

print(f"Number of unique treatment items: {len(unique_As)}")

Number of unique treatment items: 1178


In [64]:
batch_size = 4096

H_tensor = torch.as_tensor(
    windows[:, :L],
    dtype=torch.long,
)

A_tensor = torch.as_tensor(
    unique_As,
    dtype=torch.long,
    device=sasrec_model.device,
)

logits_batches = []

with torch.no_grad():
    A_emb = sasrec_model.item_embedding(A_tensor)

    for start in tqdm(range(0, len(H_tensor), batch_size)):
        end = min(start + batch_size, len(H_tensor))

        H_batch = H_tensor[start:end].to(sasrec_model.device)

        h = sasrec_model.forward(H_batch)
        last_state = h[:, -1, :]

        raw_logits = last_state @ A_emb.T
        raw_probs = torch.sigmoid(raw_logits)
        calibrated_logits = (
            beta[0]
            + beta[1] * torch.log(raw_probs)
            + beta[2] * torch.log1p(-raw_probs)
        )

        logits_batches.append(
            calibrated_logits.cpu().numpy().astype(np.float16)
        )

calibrated_logits_matrix = np.vstack(logits_batches)
print(f"Shape of calibrated_logits_matrix: {calibrated_logits_matrix.shape}")

  0%|          | 0/45 [00:00<?, ?it/s]

Shape of calibrated_logits_matrix: (183965, 1178)


# 5 Save Results 

In [65]:
output = {
    'A_to_unique_A_idx': A_to_unique_A_idx,
    'treatment_items': treatment_items,
    'future_items': future_items,
    'item_embeddings': item_embeddings,
    'calibrated_logits_matrix': calibrated_logits_matrix,
}

In [66]:
output_path = model_path / f'evaluation_data.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(output, f)